In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss

# 1. 加载数据
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']

# 2. 划分训练/验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. 标准化（可选但推荐）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

# 4. 定义并训练 Logistic Regression（baseline）
lr = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',    # 简单应对不平衡
    multi_class='multinomial',
    random_state=42
)
lr.fit(X_train_scaled, y_train)

# 5. 在验证集上评估
y_pred       = lr.predict(X_val_scaled)
y_pred_proba = lr.predict_proba(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))
print("Weighted Log Loss:", log_loss(y_val, y_pred_proba))


Accuracy: 0.6795665634674922
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.38      0.46      0.41        13
           4       0.40      0.59      0.48        46
           5       0.96      0.76      0.85       875
           6       0.87      0.89      0.88       106
           7       0.50      0.52      0.51        21
           8       0.75      0.64      0.69        98
           9       0.20      0.20      0.20         5
          10       0.77      0.68      0.72       208
          11       0.64      0.75      0.69        12
          12       0.56      0.53      0.55        88
          13       0.00      0.00      0.00        12
          14       0.13      0.50      0.20        52
          15       0.83      1.00      0.91         5
          16       0.00     

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-sc

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# 1. 加载数据
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']

# 2. 划分训练/验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. 构建 Pipeline：SMOTE → 标准化 → LogisticRegression
pipe = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=42,k_neighbors=3)),
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        multi_class='multinomial',
        random_state=42
    ))
])

# 4. 训练
pipe.fit(X_train, y_train)

# 5. 在验证集上评估
y_pred       = pipe.predict(X_val)
y_pred_proba = pipe.predict_proba(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))

def weighted_log_loss_formula(y_true, y_pred_proba, eps=1e-15):
    """
    按照公式 L = - ∑_i w_{y_i} * log(p̂_i)，w_y = 1 / f_y，
    最后除以 N 得到 average per sample。
    y_true: array-like (N,)  真实标签 0…C-1
    y_pred_proba: array (N, C)  模型预测的各类概率
    """
    y_true = np.asarray(y_true)
    N, C = y_pred_proba.shape

    # 1) 计算每个类别的频次 f_y
    class_counts = np.bincount(y_true, minlength=C)
    # 如果某类在这一集合里完全没出现，就把它当 1 处理，避免除零
    class_counts[class_counts == 0] = 1

    # 2) w_y = 1 / f_y
    class_weights = 1.0 / class_counts

    # 3) 提取每个样本的 p̂_i = P(y_true[i])
    p = y_pred_proba[np.arange(N), y_true]
    p = np.clip(p, eps, 1 - eps)  # 避免 log(0)

    # 4) 计算加权 log‐loss
    losses = - class_weights[y_true] * np.log(p)

    # 5) 返回平均 per sample
    return losses.sum() / N

# —— 一次性算 train & val loss —— 
y_train_proba = ensemble.predict_proba(X_train)
y_val_proba   = ensemble.predict_proba(X_val)

train_wll = weighted_log_loss_formula(y_train, y_train_proba)
val_wll   = weighted_log_loss_formula(y_val,   y_val_proba)

print(f"Training weighted log‐loss:   {train_wll:.4f}")
print(f"Validation weighted log‐loss: {val_wll:.4f}")


Accuracy: 0.6718266253869969
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.42      0.38      0.40        13
           4       0.40      0.59      0.47        46
           5       0.94      0.77      0.85       875
           6       0.88      0.87      0.87       106
           7       0.41      0.43      0.42        21
           8       0.75      0.68      0.72        98
           9       0.17      0.20      0.18         5
          10       0.76      0.68      0.72       208
          11       0.60      0.75      0.67        12
          12       0.54      0.50      0.52        88
          13       0.00      0.00      0.00        12
          14       0.11      0.38      0.17        52
          15       0.80      0.80      0.80         5
          16       0.00     

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-sc

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, log_loss
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

# 随机种子
RANDOM_SEED = 42

# 加载数据
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']

# 划分训练/验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

# 建立 Pipeline：ADASYN + LogisticRegression
pipe_adasyn = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=RANDOM_SEED,n_neighbors=3)),
    ('lr', LogisticRegression(
        max_iter=1000,               # 调试时可设小点，加快训练         # 之前 baseline 最优 C，可根据需要 GridSearchCV 重新调参
        random_state=RANDOM_SEED,
        multi_class='multinomial'
    ))
])

# 训练
pipe_adasyn.fit(X_train, y_train)

# 验证集评估
y_pred       = pipe_adasyn.predict(X_val)
y_pred_proba = pipe_adasyn.predict_proba(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))
print("Weighted Log Loss:", log_loss(y_val, y_pred_proba))


Accuracy: 0.6692466460268318
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.45      0.38      0.42        13
           4       0.42      0.61      0.50        46
           5       0.95      0.77      0.85       875
           6       0.85      0.85      0.85       106
           7       0.45      0.43      0.44        21
           8       0.74      0.66      0.70        98
           9       0.12      0.20      0.15         5
          10       0.76      0.67      0.71       208
          11       0.56      0.75      0.64        12
          12       0.52      0.50      0.51        88
          13       0.00      0.00      0.00        12
          14       0.12      0.44      0.19        52
          15       0.80      0.80      0.80         5
          16       0.00     

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-sc